# Using LOBSIM with Python

In [8]:
from __future__ import annotations

import datetime as dt
import hashlib
from dataclasses import dataclass
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Iterable

import pyarrow.parquet as pq
from lobsim.engine import PaperTradingSimulatorCore
from lobsim.lob_event import NormalizedLobEvent
from lobsim.replay import ReplayConfig, ReplaySession
from lobsim.sink import InMemoryLogSink
from lobsim.types import (
    Side,
    UnknownAggressorIdSentinel,
    UnknownTraderIdSentinel,
    UpdateSource,
    UpdateType,
)

### Helpers to parse sample L3 data from parquet

In [9]:
def parse_update_type(s: str) -> UpdateType:
    s = s.strip().upper()
    if s == "SNAPSHOT":
        return UpdateType.SET
    if s == "ADD":
        return UpdateType.ADD
    if s == "DELETE":
        return UpdateType.DELETE
    if s == "MATCH":
        return UpdateType.MATCH
    if s == "SUBTRACT":
        return UpdateType.SUBTRACT
    if s == "SET":
        return UpdateType.SET
    raise ValueError(f"unknown update_type: {s}")


def parse_time_us(v) -> int:
    # Matches the C++ parser: time-of-day in microseconds since midnight
    if isinstance(v, int):
        return v
    if isinstance(v, dt.time):
        return (v.hour * 3600 + v.minute * 60 + v.second) * 1_000_000 + v.microsecond
    if isinstance(v, str):
        # "HH:MM:SS.ssssss"
        hh, mm, rest = v.split(":")
        ss, frac = (rest.split(".") + ["0"])[:2]
        frac = (frac + "000000")[:6]
        return (
            int(hh) * 3600_000_000
            + int(mm) * 60_000_000
            + int(ss) * 1_000_000
            + int(frac)
        )
    raise ValueError(f"unsupported time value: {type(v)}")


def to_ticks(x: float, step: float, *, strict: bool) -> int:
    dx = Decimal(str(x))
    ds = Decimal(str(step))
    y = dx / ds
    if strict and y != y.to_integral_value():
        raise ValueError(f"value {x} not on grid {step}")
    return int(y.to_integral_value(rounding=ROUND_HALF_UP))


def stable_int64(s: str) -> int:
    h = hashlib.blake2b(s.encode("utf-8"), digest_size=8).digest()
    v = int.from_bytes(h, "little", signed=False)
    if v >= 2**63:
        v -= 2**64
    return v

### Exchange/feed-specific event schema. What the feed produces

In [10]:
@dataclass
class CoinapiRawEvent:
    ts_exchange_us: int
    ts_received_us: int
    update_type: str
    is_buy: int
    entry_px: float
    entry_sx: float
    order_id: str

### The data source. Feeds events that respect the CoinapiRawEvent schema

In [11]:
class CoinapiCoinbaseBTCUSDTSource:
    def __init__(self, path: str | Path, batch_size: int = 128):
        self._pq = pq.ParquetFile(str(path))
        self._batch_size = batch_size
        self._cols = [
            "time_exchange",
            "time_coinapi",
            "update_type",
            "is_buy",
            "entry_px",
            "entry_sx",
            "order_id",
        ]

    def __iter__(self) -> Iterable[CoinapiRawEvent]:
        for batch in self._pq.iter_batches(
            columns=self._cols, batch_size=self._batch_size
        ):
            # Access by column name for clarity
            cols = {name: batch.column(i) for i, name in enumerate(self._cols)}
            n = batch.num_rows
            for i in range(n):
                yield CoinapiRawEvent(
                    ts_exchange_us=parse_time_us(cols["time_exchange"][i].as_py()),
                    ts_received_us=parse_time_us(cols["time_coinapi"][i].as_py()),
                    update_type=str(cols["update_type"][i].as_py()),
                    is_buy=int(cols["is_buy"][i].as_py()),
                    entry_px=float(cols["entry_px"][i].as_py()),
                    entry_sx=float(cols["entry_sx"][i].as_py()),
                    order_id=str(cols["order_id"][i].as_py()),
                )

### The adapter to transform the feed schema, into the normalized event schema, required by the engine

In [12]:
class CoinapiCoinbaseBTCUSDTAdapter:
    def __init__(
        self,
        *,
        tick_size: float,
        lot_size: float,
        symbol_id: str = "BTC-USDT",
        strict: bool = True,
    ):
        self.tick_size = tick_size
        self.lot_size = lot_size
        self.symbol_id = symbol_id
        self.strict = strict

    def normalize(self, raw: CoinapiRawEvent) -> NormalizedLobEvent:
        if raw.ts_exchange_us < 0 or raw.ts_received_us < 0:
            raise ValueError("negative timestamp")
        if raw.entry_px < 0 or raw.entry_sx < 0:
            raise ValueError("negative price/size")
        if not raw.order_id:
            raise ValueError("empty order_id")

        update_type = parse_update_type(raw.update_type)
        side = Side.BUY if raw.is_buy else Side.SELL

        price_ticks = to_ticks(raw.entry_px, self.tick_size, strict=self.strict)
        qty_lots = to_ticks(raw.entry_sx, self.lot_size, strict=self.strict)

        order_id = stable_int64(raw.order_id)

        return NormalizedLobEvent(
            tsExchange=raw.ts_exchange_us,
            tsReceived=raw.ts_received_us,
            side=side,
            updateType=update_type,
            priceTicks=price_ticks,
            quantityLots=qty_lots,
            orderId=order_id,
            traderId=UnknownTraderIdSentinel,
            aggressorId=UnknownAggressorIdSentinel,
            updateSource=UpdateSource.HISTORICAL,
            symbolId=self.symbol_id,
        )


### Replay all historical events

In [13]:
engine = PaperTradingSimulatorCore()
sink = InMemoryLogSink()
engine.set_log_sink(sink)
adapter = CoinapiCoinbaseBTCUSDTAdapter(tick_size=0.01, lot_size=1e-8)

source = CoinapiCoinbaseBTCUSDTSource(
    "../sample_data/coinapi_coinbase_btcusdt_sample.parquet"
)

replay_session = ReplaySession(engine, ReplayConfig(require_monotonic_ts_received=True))
summary = replay_session.run_raw(source=source, adapter=adapter, config=ReplayConfig())
print(
    summary.first_ts_received,
    summary.last_ts_received,
    summary.num_adapter_failures,
    summary.num_raw_events,
    summary.num_normalized_events,
)


1266097 131079456 0 10000 10000


### Paper-trading / insert strategy orders to historical feed

In [14]:
engine = PaperTradingSimulatorCore()
sink = InMemoryLogSink()
engine.set_log_sink(sink)
adapter = CoinapiCoinbaseBTCUSDTAdapter(tick_size=0.01, lot_size=1e-8)

source = CoinapiCoinbaseBTCUSDTSource(
    "../sample_data/coinapi_coinbase_btcusdt_sample.parquet"
)

for i, raw_event in enumerate(source):
    norm_event = adapter.normalize(raw_event)
    engine.update(norm_event)

    if i % 500 == 0:
        top2b = engine.l2_top_n(Side.SELL, 2)
        top2a = engine.l2_top_n(Side.BUY, 2)
        a1 = engine.get_best_price_ticks(Side.SELL)

        print(f"Top 2 - BID: {top2b} | ASK: {top2a}")

        strat_event = NormalizedLobEvent(
            tsExchange=norm_event.tsExchange,
            tsReceived=norm_event.tsReceived + 1,
            side=Side.BUY,
            updateType=UpdateType.ADD,
            priceTicks=norm_event.priceTicks,
            quantityLots=2,
            orderId=123456789,
            traderId=123456789,
            aggressorId=UnknownAggressorIdSentinel,
            updateSource=UpdateSource.STRATEGY,
        )
        engine.update(strat_event)

Top 2 - BID: [] | ASK: []
Top 2 - BID: [(9355369, 1166741), (9355370, 2098699)] | ASK: [(9352797, 618361), (9352381, 2851780)]
Top 2 - BID: [(9355055, 2098699), (9355065, 2320000)] | ASK: [(9352114, 42823), (9352113, 41485)]
Top 2 - BID: [(9354732, 2320000), (9354734, 1166741)] | ASK: [(9352267, 1336089), (9351594, 1624991)]
Top 2 - BID: [(9354092, 17551), (9354093, 35102)] | ASK: [(9351270, 618361), (9350692, 76380)]
Top 2 - BID: [(9354122, 2098710), (9354727, 2320000)] | ASK: [(9352899, 618361), (9352811, 1816206)]
Top 2 - BID: [(9357653, 4276538), (9359267, 873644)] | ASK: [(9355154, 64880), (9355153, 7462852)]
Top 2 - BID: [(9358162, 2300000), (9358163, 16800000)] | ASK: [(9356401, 13928220), (9356328, 942508)]
Top 2 - BID: [(9360233, 19200000), (9362090, 75887134)] | ASK: [(9360232, 941959), (9360220, 1816206)]
Top 2 - BID: [(9361875, 2320000), (9362039, 35120)] | ASK: [(9359309, 1340709), (9359238, 941959)]
Top 2 - BID: [(9360689, 1166741), (9360690, 2058414)] | ASK: [(9358161, 1